# Mini-projet de Probabilités : GCL RANDU

### Procédure d'installation

Avant de commencer, il faut installer les modules Python suivants :

* numpy
* matplotlib
* scipy

Il faut également avoir installé Python 3.

### Auteurs

- Menez Ewen, groupe 1A
- Magassouba Ismaëla, groupe 1A

### Description

Ce projet étudie la qualité statistique du générateur congruentiel linéaire (GCL) RANDU, un algorithme historiquement utilisé dans les années 1960-1970 pour générer des nombres pseudo-aléatoires. Bien que rapide à calculer, ce générateur est connu pour présenter de graves défauts de corrélation en dimension supérieure. L'objectif est d'évaluer son comportement sur trois tailles d'échantillons ($30\,000$, $90\,000$ et $300\,000$ valeurs) à travers un test d'adéquation du $\chi^2$ et des tests spectraux en 2D et 3D.

## Importation des bibliothèques

Tout d'abord, on `import` les bibliothèques :

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chisquare

## Définition de la fonction de GCL RANDU

Un générateur congruentiel linéaire produit une suite d'entiers définie par une relation de récurrence.
Le générateur RANDU utilise les paramètres suivants : 
- Graine initiale : $x_0 = 12345$
- Multiplicateur : $a = 65539$ 
- Incrément : $c = 0$
- Module : $m = 2^{31}$

La fonction `GCL_RANDU` calcule les $n$ premiers termes entiers de la suite selon la formule $x_{k+1} = (65539 \times x_k) \bmod 2^{31}$.

In [ ]:
def GCL_RANDU(n: int, x_0: int = 12345, a:int = 65539, c:int = 0,  m: int = 2**31) -> list[int]:
    """
    Génère la liste des nombres de GCL_RANDU entre le premier élément et le n-ième élément.

    """
    randus = [0] * n
    x = x_0
    for k in range(n):
        x = (a * x + c) % m
        randus[k] = x
    return randus

## Génération des 3 échantillons

On définit tout d'abord une fonction permettant de mettre les valeurs générées entre 0 et 1.
Diviser chaque terme par $m$ applique une transformation linéaire qui ramène les entiers dans l'intervalle $[0, 1[$ tout en préservant l'uniformité de la distribution.

In [ ]:
def mettre_GCL_entre_0_et_1(nombres:list[int], m: int = 2**31) -> list[float]:
    return [elt / m for elt in nombres]

On génère ensuite les trois échantillons de valeurs normalisées dans l'intervalle $[0, 1]$ :

In [ ]:
trente_mille_donnees     =  mettre_GCL_entre_0_et_1(GCL_RANDU(30*1000))

nonante_mille_donnees    =  mettre_GCL_entre_0_et_1(GCL_RANDU(90*1000))

trois_cent_mille_donnees =  mettre_GCL_entre_0_et_1(GCL_RANDU(300*1000))

## Visualisation des 3 échantillons générés 

On représente les 3 échantillons sous forme d'histogrammes, avec en abscisse les valeurs (découpées en 10 classes) et en ordonnée le nombre d'occurrences (effectifs).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

#chaque barre mesure les effectifs entre x/10 et (x+1)/10  avec x dans {0, 1, ..., 9}
bornes = [k / 10 for k in range(11)]

def representer(ax, donnees: list[float], titre: str):
    ax.hist(donnees, bins=bornes, edgecolor='black', alpha=0.75)
    ax.axhline(len(donnees) / 10, color='red', linestyle='--', label='Effectif théorique')
    ax.legend()
    ax.set_title(titre)
    ax.set_xlabel("Valeurs dans [0, 1]")
    ax.set_ylabel("Effectifs")

representer(axes[0], trente_mille_donnees, "30 000 données")
representer(axes[1], nonante_mille_donnees, "90 000 données")
representer(axes[2], trois_cent_mille_donnees, "300 000 données")

plt.tight_layout()
plt.show()

## Évaluation statistique du générateur sur les trois échantillons

### Explication

Afin de vérifier si la suite générée simule correctement une loi uniforme continue sur $[0, 1]$, nous soumettons chaque échantillon à trois vérifications :

1. Test du $\chi^2$ (seuil de confiance $\alpha = 1\,\%$) : On découpe l'intervalle $[0, 1]$ en $k$ classes de même largeur et on compare les effectifs observés aux effectifs théoriques attendus sous une loi uniforme ($n/k$ par classe). Si la $p$-value est inférieure à $0,01$, l'hypothèse d'uniformité est rejetée.

2. Test spectral 2D : On regroupe les valeurs consécutives par paires $(u_{2k}, u_{2k+1})$ pour les représenter dans le carré $[0, 1]^2$ et vérifier l'absence de motifs ou d'alignements réguliers.

3. Test spectral 3D : On regroupe les valeurs par triplets $(u_{3k}, u_{3k+1}, u_{3k+2})$ dans le cube $[0, 1]^3$ afin de détecter d'éventuelles corrélations spatiales entre trois tirages successifs.

### Test du khi-deux avec un seuil de confiance à 1%

In [ ]:
bornes = [k / 10 for k in range(10+1)]  # [0.0, 0.1, 0.2, ..., 1.0]

arrondir = lambda x : int(x*100)/100

def test_khi_deux(donnees: list[float], alpha: float) -> str:
    effectif_observe, classes = np.histogram(donnees, bins=bornes)
    
    # Calcul des effectifs théoriques sous une loi uniforme
    nb_classes = len(classes) - 1
    nb_donnees = len(donnees)
    effectif_attendu = [ nb_donnees / nb_classes for k in range(nb_classes)]
    
    # Test du khi-deux
    stat, p_value = chisquare(effectif_observe, effectif_attendu)
    
    resultat = "VALIDÉ (loi uniforme non rejetée)" if p_value >= alpha else "REJETÉ (loi uniforme rejetée)"
    return f"Statistique chi2 = {arrondir(stat)} | p-value = {arrondir(p_value)} -> {resultat}\n"

alpha = 0.01

print("---------- TEST AVEC 30 000 DONNEES ----------")
print( test_khi_deux(trente_mille_donnees, alpha)     )
print("---------- TEST AVEC 90 000 DONNEES ----------")
print( test_khi_deux(nonante_mille_donnees, alpha)    )
print("---------- TEST AVEC 300 000 DONNEES ---------")
print( test_khi_deux(trois_cent_mille_donnees, alpha) )

On remarque alors qu'à l'exécution, le Khi-Deux et p-values sont tous deux supérieurs à $\alpha$.

### Test spectral 2D sur des doublets

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

#on considère que le nombre de données est pair.
def test_spectral_2D(ax, donnees: list[float], titre: str) -> None:
    # Découpage en paires consécutives (u_{2k}, u_{2k+1})
    x, y = donnees[0::2], donnees[1::2]
    
    ax.scatter(x, y, s=0.5, alpha=0.4)
    ax.set_title(titre)
    ax.set_xlabel("$u_{2k}$") ; ax.set_ylabel("$u_{2k+1}$")
    ax.set_xlim(0, 1) ; ax.set_ylim(0, 1)

test_spectral_2D(axes[0], trente_mille_donnees, "30 000 données (15 000 doublets)")
test_spectral_2D(axes[1], nonante_mille_donnees, "90 000 données (45 000 doublets)")
test_spectral_2D(axes[2], trois_cent_mille_donnees, "300 000 données (150 000 doublets)")

plt.show()

On observe que la répartition des doublets semble uniforme, et aucun schéma visuel ne semble être visible.

### Test spectral 3D sur des triplets 

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), subplot_kw={'projection': '3d'})

def test_spectral_3D(ax, donnees: list[float], titre: str) -> None:
    # Découpage en triplets consécutifs (u_{3k}, u_{3k+1}, u_{3k+2})
    x, y, z = donnees[0::3], donnees[1::3], donnees[2::3]
    
    ax.scatter(x, y, z, s=0.5, alpha=0.4)
    ax.set_title(titre)
    ax.set_xlabel("$u_{3k}$") ; ax.set_ylabel("$u_{3k+1}$") ; ax.set_zlabel("$u_{3k+2}$")
    ax.set_xlim(0, 1) ; ax.set_ylim(0, 1) ; ax.set_zlim(0, 1)
    
    # Orientation de la caméra dans l'axe des plans parallèles de RANDU
    ax.view_init(elev=10, azim=53)

test_spectral_3D(axes[0], trente_mille_donnees, "30 000 données (10 000 triplets)")
test_spectral_3D(axes[1], nonante_mille_donnees, "90 000 données (30 000 triplets)")
test_spectral_3D(axes[2], trois_cent_mille_donnees, "300 000 données (100 000 triplets)")

plt.show()

On observe cette fois-ci qu'un schéma visuel apparaît de manière évidente : les données sont regroupées en tranches, sans aucune valeur entre ces tranches.

## Conclusion

Voici le bilan des tests statistiques effectués sur le générateur congruentiel linéaire RANDU ($x_0 = 12345$, $a = 65539$, $c = 0$, $m = 2^{31}$) :

| Test effectué | 30 000 données | 90 000 données | 300 000 données |
| :--- | :--- | :--- | :--- |
| Test du $\chi^2$ (seuil à 1 %) | Validé ($0.65 \ge 0,01$) | Validé ($0.45 \ge 0,01$) | Validé ($0.78 \ge 0,01$) |
| Test spectral 2D (doublets) | Validé (répartition homogène sur $[0, 1]^2$) | Validé (répartition homogène sur $[0, 1]^2$) | Validé (répartition homogène sur $[0, 1]^2$) |
| Test spectral 3D (triplets) | Rejeté (alignement visible sur 15 plans) | Rejeté (15 plans très marqués) | Rejeté (15 plans flagrants) |

*Note : p est arrondi à $10^{-2}$ près.*

### Synthèse et interprétation

1. **Test du Khi-deux** : Sur les trois tailles d'échantillons, RANDU semble produire des nombres qui suivent une loi uniforme sur $[0 ; 1]$.

2. **Test spectral en 2D** : Là aussi, RANDU semble produire des nombres qui sont répartis uniformément entre $0$ et $1$.

3. **Test spectral en 3D** : Cette fois, le cube semble contenir des points répartis de manière non-uniforme : on distingue 15 plans distincts. Cela indique qu'il y a une relation entre trois nombres générés par GCL RANDU, et qu'en ayant connaissance des deux premiers on puisse s'attendre à ce que le troisième ait une certaine valeur. Cela indique également qu'il y a de vastes intervalles entre les plans dans lesquels il n'y a aucune valeur, donc qui ne sont pas "couverts" par le troisième nombre.

On en déduit donc que le générateur de nombres pseudo-aléatoires GCL-RANDU passe les tests d'uniformité unidimensionnelle (Khi-deux) et bidimensionnelle (Spectral 2D), mais échoue au test tridimensionnel (Spectral 3D). Cela indique que ce générateur ne sera pas fiable tel quel, et que RANDU est structurellement défaillant en dimension $\ge 3$ et doit être proscrit pour toute simulation multidimensionnelle.